# Créer un tenseur : valeurs, shape, dtype et device

Reconnaître une case, une ligne et un tableau avant de les utiliser.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#tenseur-forme-type-device).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous fabriquons trois observations pour comprendre le programme. Chaque ligne est un extrait ; ses colonnes sont la RMS et le facteur de crête.

| Ligne | Extrait et cible | RMS choisie | Facteur de crête choisi |
| --- | --- | --- | --- |
| 0 | Voix A → 0 | 0,10 | 1,5 |
| 1 | Percussion B → 1 | 0,20 | 4,5 |
| 2 | Voix C → 0 | 0,15 | 2,5 |

Ces descripteurs sont choisis à la main. Aucun fichier audio n’a été analysé pour fabriquer ce tableau.

La cible est une annotation par extrait : 0 signifie voix et 1 percussion. Elle n’est pas fournie au modèle comme une mesure sonore.

| Écriture | Sens ici |
| --- | --- |
| torch.tensor(...) | Créer un tenseur à partir des nombres fournis. |
| dtype=torch.float32 | Stocker les mesures en nombres flottants sur 32 bits. |
| dtype=torch.long | Stocker les classes comme entiers signés sur 64 bits : long est un alias de int64. |
| X.shape | Lire les tailles des axes ; ici trois extraits et deux mesures. |
| X[0] et X[0,1] | Lire la première ligne, puis la deuxième valeur de cette ligne. |

Le nom long ne décrit pas la longueur du tableau. Le type entier préparera le mode à indices de classe de CrossEntropyLoss.

X a la forme (3,2). X[0] retire l’axe des extraits : il reste deux mesures, de forme (2,). X[0,1] sélectionne une seule valeur : 1,5.

Le tuple (2,) contient une seule taille ; sa virgule distingue ce tuple du nombre 2 entre parenthèses. La forme () indique qu’aucun axe ne reste.

X.device indique où le tenseur est stocké. Ici, cpu désigne le processeur principal de l’ordinateur : les données et les calculs restent sur ce CPU.

torch.isfinite(X) teste chaque case : True signifie « valeur finie », sans infini ni NaN. NaN indique une valeur numérique indéfinie ; ce contrôle aide à repérer un calcul invalide.

finies.all() vérifie ensuite si toutes les cases ont passé le test. Il produit un tenseur scalaire ; item() en extrait ici un booléen Python, utilisable dans un if. Sur valeur, item() extrait de même le nombre 1,5.

## Démonstration 1 · Créer, sélectionner, puis contrôler

Relier chaque résultat à une partie du tableau.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

X stocke les mesures continues ; cibles stocke trois indices de classe. Les deux objets ne sont pas interchangeables.

In [4]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]], dtype=torch.float32)
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [5]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

L’indice zéro désigne la première ligne. Le couple [0,1] signifie ligne zéro, colonne un. Lire ces positions ne modifie pas X.

In [6]:
premier = X[0]
valeur = X[0, 1]


In [7]:
observer('premier', premier)
observer('valeur', valeur)

premier forme (2,) type torch.float32 requires_grad False
  valeurs : [0.10000000149011612, 1.5]
valeur forme () type torch.float32 requires_grad False
  valeurs : 1.5


### Calculer l’état suivant

isfinite renvoie un booléen par case. True signifie que la valeur n’est ni NaN ni un infini.

In [8]:
finies = torch.isfinite(X)


In [9]:
observer('finies', finies)

finies forme (3, 2) type torch.bool requires_grad False
  valeurs : [[True, True], [True, True], [True, True]]


### Calculer l’état suivant

all() demande si tous les booléens valent True. item() extrait ensuite ce seul résultat dans un booléen Python.

Cette écriture compacte enchaîne les trois instructions séparées ci-dessus. On contrôle les cases, on les réduit, puis on extrait la valeur.

In [10]:
toutes_finies = finies.all()
controle = toutes_finies.item()


In [11]:
observer('toutes_finies', toutes_finies)
observer('controle', controle)

toutes_finies forme () type torch.bool requires_grad False
  valeurs : True
controle = True


### Lire les sorties

dtype et device décrivent l’objet ; ils ne changent pas ses valeurs. Tous les calculs restent sur CPU.

In [12]:
print("tableau", X.tolist(), "forme", tuple(X.shape))
print("ligne", premier.tolist(), "forme", tuple(premier.shape))
print("case", valeur.item(), "forme", tuple(valeur.shape))
print("types", X.dtype, cibles.dtype, "long=int64", torch.long == torch.int64)


tableau [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]] forme (3, 2)
ligne [0.10000000149011612, 1.5] forme (2,)
case 1.5 forme ()
types torch.float32 torch.int64 long=int64 True


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("finies", finies.tolist(), "forme", tuple(finies.shape))
print("tout", toutes_finies.item(), "forme", tuple(toutes_finies.shape))
print("controle", controle, "Python", type(controle).__name__, "device", X.device)

finies [[True, True], [True, True], [True, True]] forme (3, 2)
tout True forme ()
controle True Python bool device cpu


Une case, une ligne et le tableau entier conservent des formes distinctes. Les tests de finitude lisent les données sans les corriger.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Choisissez le dtype des indices de classes [0,1,0] pour CrossEntropyLoss dans son mode à indices.

In [14]:
exo_valeurs = [0,1,0]
exo_dtype = ...  # Remplacer les points par votre réponse.
if exo_dtype is Ellipsis or exo_dtype is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_cibles = torch.tensor(exo_valeurs,dtype=exo_dtype)
    assert exo_cibles.dtype == torch.int64
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

long est un autre nom de int64. Cela ne signifie pas que toutes les cibles de toutes les tâches sont entières.

</details>

### Exercice 2

Le calcul complet isfinite(...).all().item() renverra-t-il True ici ?

In [15]:
exo_x = torch.tensor([1.,float("nan"),2.])
exo_fini = ...  # Remplacer les points par votre réponse.
if exo_fini is Ellipsis or exo_fini is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_fini is torch.isfinite(exo_x).all().item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

isfinite teste chaque case ; all demande que toutes soient valides ; item extrait le booléen scalaire.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_valeurs = [0,1,0]
exo_dtype = torch.long
if exo_dtype is Ellipsis or exo_dtype is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_cibles = torch.tensor(exo_valeurs,dtype=exo_dtype)
    assert exo_cibles.dtype == torch.int64
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([1.,float("nan"),2.])
exo_fini = False
if exo_fini is Ellipsis or exo_fini is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_fini is torch.isfinite(exo_x).all().item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [16]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()